# Bernstein–Vazirani (BV) Assignment

Goal: discover a hidden $n$-bit string $s$ using one oracle call.

**Concept**

- Oracle returns $f_s(x) = s \cdot x \pmod 2$.
- Build superposition, call oracle once, then decode with Hadamards.
- Measurement of inputs yields $s$ in one query.

**Circuit Construction**

- Classical queries needed: $n$ (e.g., $n=4\Rightarrow4$, $n=6\Rightarrow6$).
- Inputs: $n$ qubits; ancilla prepared to $| - \rangle$.
- Oracle: CNOT from input $i$ to ancilla if $s_i=1$.
- Second $H$ layer: decodes phase to bits; measure inputs.

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def bv_circuit(n, s):
    qc = QuantumCircuit(n + 1, n)
    qc.h(range(n))
    qc.x(n)
    qc.h(n)
    for i, b in enumerate(s):
        if b == '1':
            qc.cx(i, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

n = 4
s = '1011'
qc = bv_circuit(n, s)
print(qc.draw())
sim = AerSimulator()
result = sim.run(transpile(qc, sim), shots=4096).result()
counts = result.get_counts()
print(counts)
print('argmax:', max(counts, key=counts.get))


     ┌───┐          ┌───┐          ┌─┐           
q_0: ┤ H ├───────■──┤ H ├──────────┤M├───────────
     ├───┤┌───┐  │  └┬─┬┘          └╥┘           
q_1: ┤ H ├┤ H ├──┼───┤M├────────────╫────────────
     ├───┤└───┘  │   └╥┘      ┌───┐ ║      ┌─┐   
q_2: ┤ H ├───────┼────╫────■──┤ H ├─╫──────┤M├───
     ├───┤       │    ║    │  └───┘ ║ ┌───┐└╥┘┌─┐
q_3: ┤ H ├───────┼────╫────┼────■───╫─┤ H ├─╫─┤M├
     ├───┤┌───┐┌─┴─┐  ║  ┌─┴─┐┌─┴─┐ ║ └───┘ ║ └╥┘
q_4: ┤ X ├┤ H ├┤ X ├──╫──┤ X ├┤ X ├─╫───────╫──╫─
     └───┘└───┘└───┘  ║  └───┘└───┘ ║       ║  ║ 
c: 4/═════════════════╩═════════════╩═══════╩══╩═
                      1             0       2  3 
{'1101': 4096}
argmax: 1101


Expected: argmax equals the secret bitstring $s$.

In [1]:
%matplotlib inline
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

def bv_circuit(n, s):
    qc = QuantumCircuit(n + 1, n)
    qc.h(range(n))
    qc.x(n)
    qc.h(n)
    for i, b in enumerate(s):
        if b == '1':
            qc.cx(i, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

n = 4
s = '1011'
qc = bv_circuit(n, s)

# 🔵 Inline circuit diagram
qc.draw(output='mpl', style={'textcolor': None})



sim = AerSimulator()
result = sim.run(transpile(qc, sim), shots=4096).result()
counts = result.get_counts()
counts, max(counts, key=counts.get)


ModuleNotFoundError: No module named 'matplotlib'

In [5]:
# Bernstein–Vazirani Algorithm in Qiskit

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram


def bv_oracle(s: str) -> QuantumCircuit:
    """
    Oracle for the BV algorithm.
    Encodes f(x) = s · x (mod 2) using CNOTs controlled by bits of s.

    Qubits:
      - q[0..n-1] : input register
      - q[n]      : ancilla (phase kickback)
    """
    n = len(s)
    oracle = QuantumCircuit(n + 1)

    # For each bit of s that is '1', apply CNOT from that qubit to ancilla
    for i, bit in enumerate(s):
        if bit == '1':
            oracle.cx(i, n)

    return oracle


def bv_circuit(s: str) -> QuantumCircuit:
    """
    Builds the full BV circuit for a given hidden string s.
    """
    n = len(s)
    qc = QuantumCircuit(n + 1, n)  # n+1 qubits (n inputs + 1 ancilla), n classical bits

    # 1) Prepare |0...0⟩|1⟩
    qc.x(n)  # put ancilla in |1⟩

    # 2) Apply Hadamard to all qubits (inputs + ancilla)
    qc.h(range(n + 1))

    # 3) Apply the oracle
    oracle = bv_oracle(s)
    qc.compose(oracle, inplace=True)

    # 4) Apply Hadamard again to the input qubits
    qc.h(range(n))

    # 5) Measure input qubits (they will contain s)
    qc.measure(range(n), range(n))

    return qc


def run_bv(s: str, shots: int = 1024):
    """
    Builds and runs the BV circuit for hidden string s,
    then prints the recovered string.
    """
    qc = bv_circuit(s)

    # Optional: draw the circuit (uncomment in a notebook / VS Code with matplotlib set up)
    qc.draw()

    simulator = AerSimulator()
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc, shots=shots).result()
    counts = result.get_counts()

    print("Measurement results:", counts)

    # The bitstring with the highest count is our estimate of s
    recovered_s = max(counts, key=counts.get)
    print(f"Hidden string s       : {s}")
    print(f"Recovered string (BV) : {recovered_s}")


if __name__ == "__main__":
    # Example: n = 4, s = "1011"
    hidden_s = "1011"
    run_bv(hidden_s)
    

Measurement results: {'1101': 1024}
Hidden string s       : 1011
Recovered string (BV) : 1101
